# Problem Book P1, Chapter 7: Generating Functions Meet Arithmetic (Python companion)

Part A checks the numbers of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. Only `numpy`, `sympy`, `mpmath` and `matplotlib` are needed.

*Before running a cell, predict its output.*

In [ ]:
import math
from math import comb, gcd, log, sqrt, pi
from functools import lru_cache
import numpy as np
from sympy import divisors, factorint, fibonacci, primerange, mobius, divisor_count, divisor_sigma
import mpmath as mp
import matplotlib.pyplot as plt

N = 600
def parts(S, N=N):                      # number of partitions of n into parts from S, n <= N
    a = [1] + [0] * N
    for k in S:
        for n in range(k, N + 1): a[n] += a[n - k]
    return a
p = parts(range(1, N + 1))
def digits(n, b):
    r = []
    while n: r.append(n % b); n //= b
    return r
def v(n, q):
    k = 0
    while n % q == 0: n //= q; k += 1
    return k
print(p[:25], p[200])

## Part A: checks

**Problem 7.1 (parts 1, 2, 3).** The nearest-integer formula, and integer triangles of perimeter $n$.

In [ ]:
a = parts([1, 2, 3])
print(all(a[n] == round((n + 3)**2 / 12) for n in range(N + 1)), all(a[n] - a[n - 6] == n for n in range(6, N + 1)))
def T(n): return sum(1 for x in range(1, n) for y in range(x, n) if n - x - y >= y and x + y > n - x - y)
print(all(T(2 * m + 3) == a[m] for m in range(40)), all(T(2 * m) == (a[m - 3] if m >= 3 else 0) for m in range(1, 40)))

**Problem 7.2 (parity by reflection).** $p(n)\equiv sc(n)$, and the Durfee-square identity for $\prod(1+x^{2k-1})$.

In [ ]:
d = [1] + [0] * N                         # distinct odd parts
for k in range(1, N + 1, 2):
    for n in range(N, k - 1, -1): d[n] += d[n - k]
print(all((p[n] - d[n]) % 2 == 0 for n in range(N + 1)))
M = 80; rhs = [0] * (M + 1); k = 0
while k * k <= M:
    q = [0] * (M + 1); q[k * k] = 1
    for j in range(1, k + 1):             # divide by (1 - x^{2j})
        for n in range(2 * j, M + 1): q[n] += q[n - 2 * j]
    rhs = [x + y for x, y in zip(rhs, q)]; k += 1
print(rhs == d[:M + 1], "  p(n) odd for", sum(x % 2 for x in p), "of the", N + 1, "values n <= 600")

**Problem 7.3 (Glaisher).** Both counts for $k=2,\dots,5$, and the bijection on $7+2+2+2+2+1+1+1$.

In [ ]:
def rep_less(k, N):
    A = [1] + [0] * N
    for j in range(1, N + 1):
        B = [0] * (N + 1)
        for n in range(N + 1):
            if A[n]:
                for m in range(k):
                    if n + m * j > N: break
                    B[n + m * j] += A[n]
        A = B
    return A
print(all(parts([j for j in range(1, 151) if j % k], 150) == rep_less(k, 150) for k in (2, 3, 4, 5)))
def glaisher(lam, k):
    out = []
    for j in set(lam):
        for i, a in enumerate(digits(lam.count(j), k)): out += [j * k**i] * a
    return sorted(out, reverse=True)
print(glaisher([7, 2, 2, 2, 2, 1, 1, 1], 3))

**Problem 7.4 (Stanley, Elder).** Total number of $k$'s against part sizes appearing at least $k$ times.

In [ ]:
def gen(n, m=None):
    if m is None: m = n
    if n == 0: yield []; return
    for k in range(min(n, m), 0, -1):
        for r in gen(n - k, k): yield [k] + r
print(all(sum(l.count(k) for l in gen(n)) == sum(1 for l in gen(n) for m in set(l) if l.count(m) >= k)
          for n in range(1, 22) for k in (1, 2, 3)))

**Problem 7.5 (logarithmic derivative).** $np(n)=\sum\sigma(k)p(n-k)$, and Gauss's formula for $q=2,3$.

In [ ]:
print(all(n * p[n] == sum(int(divisor_sigma(k)) * p[n - k] for k in range(1, n + 1)) for n in range(1, 200)))
Mq = lambda d, q: sum(int(mobius(d // e)) * q**e for e in divisors(d)) // d
print(all(sum(d * Mq(d, q) for d in divisors(k)) == q**k for q in (2, 3) for k in range(1, 15)))

**Problem 7.6 (Kolberg).** The gaps between generalised pentagonal numbers, which a periodic sequence could not have.

In [ ]:
pent = sorted({j * (3 * j - 1) // 2 for j in range(-30, 31)})
print(pent[:12], [b - a for a, b in zip(pent, pent[1:])][:12])

**Problem 7.7 (Ramanujan).** The congruences, Jacobi's identity, and the claim of (d) on $xE(x)^4$.

In [ ]:
print(all(p[5 * n + 4] % 5 == 0 for n in range(119)), all(p[7 * n + 5] % 7 == 0 for n in range(85)),
      all(p[11 * n + 6] % 11 == 0 for n in range(54)))
def E_pow(r, N=N):                         # coefficients of E(x)^r
    c = [1] + [0] * N
    for k in range(1, N + 1):
        for _ in range(r):
            for n in range(N, k - 1, -1): c[n] -= c[n - k]
    return c
E3 = E_pow(3, 300); jac = [0] * 301
for k in range(30):
    if k * (k + 1) // 2 <= 300: jac[k * (k + 1) // 2] = (-1)**k * (2 * k + 1)
print("Jacobi:", E3 == jac)
xE4 = [0] + E_pow(4, 299)
print("x E^4 at multiples of 5:", all(xE4[n] % 5 == 0 for n in range(0, 300, 5)))

**Problem 7.8 (Kummer).** Carries, the bound $p^{v_p\binom nk}\le n$, and $\binom nk\mid\operatorname{lcm}(1,\dots,n)$.

In [ ]:
def carries(m, n, q):
    c = k = 0
    while m or n or c:
        c = 1 if m % q + n % q + c >= q else 0; k += c; m //= q; n //= q
    return k
print(all(v(comb(m + n, m), q) == carries(m, n, q) for q in (2, 3, 5) for m in range(60) for n in range(60)))
print(all(math.lcm(*range(1, n + 1)) % comb(n, k) == 0 for n in range(1, 120) for k in range(n + 1)))

**Problem 7.9 (odd Catalan numbers).** $v_2(C_n)=s(n+1)-1$.

In [ ]:
cat = [comb(2 * n, n) // (n + 1) for n in range(400)]
print([n for n in range(400) if cat[n] % 2], all(v(cat[n], 2) == bin(n + 1).count("1") - 1 for n in range(400)))

**Problem 7.10 (modulo 3).** The counts of (a) and (b), and the totals of (c) for $k=5$.

In [ ]:
ok = True
for n in range(400):
    row = [comb(n, k) % 3 for k in range(n + 1)]; dg = digits(n, 3)
    ok &= row.count(1) + row.count(2) == 2**dg.count(1) * 3**dg.count(2) and row.count(1) - row.count(2) == 2**dg.count(1)
k = 5; rows = [[comb(n, j) % 3 for j in range(n + 1)] for n in range(3**k)]
c = [sum(r.count(t) for r in rows) for t in (0, 1, 2)]
print(ok, c, [3**k * (3**k + 1) // 2 - 6**k, (6**k + 4**k) // 2, (6**k - 4**k) // 2])

**Problem 7.11 (modulo 4).** Entries $\equiv2$ against $B(n)2^{s(n)-1}$.

In [ ]:
def B10(n): b = bin(n)[2:]; return sum(1 for i in range(len(b) - 1) if b[i:i + 2] == "10")
print(all(sum(1 for k in range(n + 1) if comb(n, k) % 4 == 2) == B10(n) * 2**(bin(n).count("1") - 1) for n in range(1, 500)))
print("rows with no entry = 2 mod 4:", [n for n in range(1, 300) if B10(n) == 0])

**Problem 7.12 ($\binom{2n}n$ modulo $p$).** The product formula and the proportion of $n<p^4$ with $p\nmid\binom{2n}n$.

In [ ]:
print(all((comb(2 * n, n) - math.prod(comb(2 * t, t) for t in digits(n, q))) % q == 0 for q in (3, 5, 7, 11) for n in range(500)))
print([(q, sum(1 for n in range(q**4) if comb(2 * n, n) % q) / q**4, ((q + 1) / (2 * q))**4) for q in (3, 5)])

**Problem 7.13 (Touchard).** $B_{n+p}\equiv B_n+B_{n+1}$, with Bell numbers from the Bell triangle.

In [ ]:
bell, row = [1], [1]
for i in range(80):
    new = [row[-1]]
    for x in row: new.append(new[-1] + x)
    row = new; bell.append(row[0])
print(bell[:9], all((bell[n + q] - bell[n] - bell[n + 1]) % q == 0 for q in (2, 3, 5, 7, 11, 13) for n in range(60)))

**Problem 7.14 (Stern).** Hyperbinary counts, row sums, coprimality.

In [ ]:
S = [0, 1]
for n in range(2, 5000): S.append(S[n // 2] if n % 2 == 0 else S[n // 2] + S[n // 2 + 1])
@lru_cache(None)
def h(n, j=0):
    if n == 0: return 1
    if 2**j > n: return 0
    return sum(h(n - c * 2**j, j + 1) for c in range(3) if c * 2**j <= n)
print(all(S[n + 1] == h(n) for n in range(400)), all(sum(S[2**k:2**(k + 1)]) == 3**k for k in range(12)),
      all(gcd(S[n], S[n + 1]) == 1 for n in range(4999)))

**Problem 7.15 (Stern, Pascal, Fibonacci).** Carlitz's count, the maxima, where they are reached, and the symmetry of the leads.

In [ ]:
print(all(S[n + 1] == sum(1 for k in range(n + 1) if comb(n - k, k) % 2) for n in range(400)))
print([(k, max(S[2**k:2**(k + 1) + 1]), int(fibonacci(k + 2)), (2**(k + 2) - (-1)**k) // 3,
        [n for n in range(2**k, 2**(k + 1) + 1) if S[n] == fibonacci(k + 2)]) for k in range(1, 9)])
print(all(S[3 * 2**k - n] == S[n] for k in range(1, 11) for n in range(2**k, 2**(k + 1) + 1)))

**Problem 7.16 (binary partitions).** Recurrences, parity, the claim of the leads that $8\nmid b(n)$, and the growth.

In [ ]:
M = 4000; b = parts([2**j for j in range(13)], M)
print(b[:12], all(b[2 * n + 1] == b[2 * n] and b[2 * n] == b[2 * n - 1] + b[n] for n in range(1, M // 2)))
print(all(b[n] % 2 == 0 for n in range(2, M + 1)), "divisible by 8:", [n for n in range(M + 1) if b[n] % 8 == 0])
print([(n, round(log(b[n]) / (log(n)**2 / (2 * log(2))), 3)) for n in (64, 512, 4000)], "(ratio tends to 1 slowly)")

**Problem 7.17 (Fibonacci divisibility).** Ranks of apparition, the claim $z(m)\le2m$, and the Jacobsthal numbers.

In [ ]:
def z(m):
    a, b2, n = 0, 1, 0
    while True:
        a, b2 = b2, (a + b2) % m; n += 1
        if a == 0: return n
print(max(z(m) / m for m in range(2, 2000)), [m for m in range(2, 2000) if z(m) == 2 * m])
J = [0, 1]
for _ in range(60): J.append(J[-1] + 2 * J[-2])
print(J[:10], all(x % 2 for x in J[1:]))

**Problem 7.18 (Thue–Morse).** The quadratic equation over $\mathbb F_2$, with polynomials as Python integers (bit $i$ = coefficient of $x^i$).

In [ ]:
L = 400
t = sum((bin(n).count("1") % 2) << n for n in range(L))
def mul2(a, b):                          # product in F_2[x], truncated to degree < L
    r = 0
    while b:
        if b & 1: r ^= a
        a <<= 1; b >>= 1
    return r & ((1 << L) - 1)
one_x = 0b11
lhs = mul2(mul2(mul2(one_x, one_x), one_x), mul2(t, t)) ^ mul2(mul2(one_x, one_x), t) ^ 0b10
print("equation holds to order 400:", lhs == 0)

**Problem 7.19 (ordered factorisations).** $H(12)$, $H(p_1\cdots p_k)$, the bound $H(n)\le n^\rho$, and Kalmár's constant.

In [ ]:
X = 20000
H = [0, 1] + [0] * (X - 1)
for d in range(1, X + 1):                 # H(n) = sum of H(d) over proper divisors d
    for m in range(2 * d, X + 1, d): H[m] += H[d]
mp.mp.dps = 20
rho = mp.findroot(lambda s: mp.zeta(s) - 2, 1.73)
print(H[12], [H[n] for n in (2, 6, 30, 210)], float(rho), all(H[n] <= n**float(rho) for n in range(1, X + 1)))
print(sum(H) / X**float(rho), float(-1 / (rho * mp.zeta(rho, derivative=1))))

**Problem 7.20 ($\sum d(n)^2$).** The Euler-product identity on the first coefficients, and the slow approach to $1/\pi^2$.

In [ ]:
X = 10**5
dn = np.zeros(X + 1, dtype=np.int64)
for k in range(1, X + 1): dn[k::k] += 1
t4 = np.ones(X + 1, dtype=np.int64); t4[0] = 0
for _ in range(3):                         # tau_4 = 1*1*1*1 by Dirichlet convolution
    new = np.zeros(X + 1, dtype=np.int64)
    for k in range(1, X + 1): new[k::k] += t4[1:X // k + 1]
    t4 = new
conv = np.zeros(X + 1, dtype=np.int64)
for m in range(1, int(sqrt(X)) + 1):
    mu = int(mobius(m))
    if mu: conv[m * m::m * m] += mu * t4[1:X // (m * m) + 1]
print("d^2 = tau_4 * g:", np.array_equal(conv[1:], dn[1:]**2))
for x in (10**3, 10**4, 10**5):
    print(x, (dn[1:x + 1]**2).sum() / (x * log(x)**3), 1 / pi**2)

**Problem 7.21 (powerful numbers).** The count against $\frac{\zeta(3/2)}{\zeta(3)}\sqrt x+\frac{\zeta(2/3)}{\zeta(2)}x^{1/3}$, and the Pell pairs.

In [ ]:
X = 10**6
pw = np.zeros(X + 1, dtype=bool)
sqf = np.ones(X + 1, dtype=bool)
for q in primerange(2, int(X**0.5) + 1): sqf[q * q::q * q] = False
for bb in range(1, int(round(X**(1 / 3))) + 2):
    if bb**3 <= X and sqf[bb]:
        for aa in range(1, int(sqrt(X / bb**3)) + 1): pw[aa * aa * bb**3] = True
for x in (10**4, 10**5, 10**6):
    print(x, pw[1:x + 1].sum(), round(float(mp.zeta(1.5) / mp.zeta(3)) * sqrt(x) + float(mp.zeta(2 / 3) / mp.zeta(2)) * x**(1 / 3), 1))
X2, Y2, pairs = 3, 1, []
for _ in range(5):
    pairs.append((8 * Y2 * Y2, X2 * X2)); X2, Y2 = 3 * X2 + 8 * Y2, X2 + 3 * Y2
print(pairs, all(all(e >= 2 for e in factorint(u).values()) for u, w in pairs))

**Problem 7.22 (abelian groups).** $A(n)$ from partitions of the exponents, and the average.

In [ ]:
X = 10**5
A = [0] + [math.prod(p[e] for e in factorint(n).values()) for n in range(1, X + 1)]
c = float(mp.nprod(lambda k: mp.zeta(k), [2, mp.inf]))
print(A[360], A[2**10], [round(sum(A[1:x + 1]) / x, 4) for x in (10**3, 10**4, 10**5)], c)
c2 = float(mp.zeta(0.5) * mp.nprod(lambda j: mp.zeta(j / 2), [3, mp.inf]))
c3 = float(mp.zeta(1 / 3) * mp.zeta(2 / 3) * mp.nprod(lambda j: mp.zeta(j / 3), [4, mp.inf]))
print("c2, c3 =", round(c2, 2), round(c3, 2), " with the x^(1/2) and x^(1/3) terms:", round(c + c2 / sqrt(X) + c3 / X**(2 / 3), 4))

## Part B: laboratory

**L7.1 (Dyson's rank).** The rank of a partition is its largest part minus its number of parts. Dyson guessed that the
partitions of $5n+4$ fall into five equal classes by rank modulo $5$, and those of $7n+5$ into seven by rank modulo $7$. Check it,
and show that the rank fails for $11$.

**L7.2 (Ramanujan's most beautiful identity).** Check $\sum_np(5n+4)x^n=5\frac{E(x^5)^5}{E(x)^6}$ to many terms.

**L7.3 (the period of Bell numbers modulo $p$).** By Touchard's congruence, $(B_n\bmod p)$ is periodic. Find the minimal period for
$p=2,3,5$ and compare with $\frac{p^p-1}{p-1}$.

**L7.4 (the parity of $p(n)$).** Plot the proportion of odd values of $p(n)$ for $n\le N$, up to $N=20000$, using
$\sum p(n)x^n\equiv\prod(1+x^{2k-1})\pmod2$ (Problem 7.2) or the pentagonal recurrence modulo $2$.

*Try them before reading on.*

### Solutions
### L7.1

In [ ]:
from collections import Counter
def rank_counts(n, m):
    c = Counter()
    for l in gen(n): c[(l[0] - len(l)) % m] += 1
    return [c[r] for r in range(m)]
print("5n+4:", [rank_counts(n, 5) for n in (4, 9, 14, 19)])
print("7n+5:", [rank_counts(n, 7) for n in (5, 12, 19)])
print("11n+6, rank mod 11:", rank_counts(17, 11))

The rank splits evenly for $5$ and $7$ (Atkin and Swinnerton-Dyer, 1954), but not for $11$: for $n=17$ the classes have unequal
sizes. Andrews and Garvan's \emph{crank} (1988) does the job for all three moduli.

### L7.2

In [ ]:
K = 110
E5 = [0] * (5 * K + 1)
e1 = E_pow(1, K)
for i, c in enumerate(e1):
    if 5 * i <= 5 * K: E5[5 * i] = c                # E(x^5)
def mulser(a, b, K):
    return [sum(a[i] * b[n - i] for i in range(n + 1)) for n in range(K + 1)]
num = [1] + [0] * K
for _ in range(5): num = mulser(num, E5, K)
inv6 = parts(range(1, K + 1), K)                     # 1/E(x)
for _ in range(5): inv6 = mulser(inv6, parts(range(1, K + 1), K), K)
rhs = [5 * x for x in mulser(num, inv6, K)]
print(all(rhs[n] == p[5 * n + 4] for n in range(K + 1)))

The identity holds to $110$ terms, and it shows at once that $5\mid p(5n+4)$. Its proofs use the modular equation of degree $5$ for
the function $E(x)$, the beginning of the theory of modular forms (Tome~5, Chapter~12).

### L7.3

In [ ]:
def bell_mod(q, L):
    row, out = [1], [1]
    for _ in range(L):
        new = [row[-1]]
        for x in row: new.append((new[-1] + x) % q)
        row = new; out.append(row[0])
    return out
for q in (2, 3, 5):
    target = (q**q - 1) // (q - 1)
    seq = bell_mod(q, 2 * target + 20)
    per = next(P for P in range(1, target + 1) if all(seq[n + P] == seq[n] for n in range(target + 20)))
    print(q, per, target)

The minimal periods are $3$, $13$ and $781$, exactly $\frac{p^p-1}{p-1}$ each time (for $p=7$ the candidate is $137257$, too long
for this simple loop). Iterating Touchard's congruence shows that
the period divides this number; that it is always equal to it is a conjecture checked for many primes.

### L7.4

In [ ]:
Nmax = 20000
par = np.zeros(Nmax + 1, dtype=np.uint8); par[0] = 1
for k in range(1, Nmax + 1, 2):                  # multiply by (1 + x^k) modulo 2
    par[k:] ^= par[:-k].copy()
prop = np.cumsum(par) / np.arange(1, Nmax + 2)
print([round(prop[n], 4) for n in (100, 1000, 10000, 20000)])
plt.figure(figsize=(6, 3)); plt.plot(prop); plt.axhline(0.5, color="gray", ls="--")
plt.xlabel("N"); plt.ylabel("proportion of odd p(n), n <= N"); plt.ylim(0.4, 0.6); plt.show()

The proportion hovers very close to $\frac12$, as Parkin and Shanks conjectured in 1967; yet nobody can prove that it stays
above any fixed positive number. Note the trick: $(1+x^k)$ modulo $2$ is a shift and an exclusive or.